# Customer Risk Prediction
**End-to-end ML pipeline**: EDA → Preprocessing → Modelling → Evaluation

**Author:** Mohammad Zaid  
**Stack:** Python · scikit-learn · Pandas · Matplotlib · Seaborn

## 0. Imports & Config

In [ ]:
import sys
sys.path.append('../src')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
RANDOM_STATE = 42
print('All imports OK')

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/raw/customer_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Stats ===')
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Target class distribution
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['risk_label'].value_counts()
ax.bar(['Low Risk (0)', 'High Risk (1)'], counts.values, color=['#4CAF50', '#F44336'])
ax.set_title('Target Class Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Class balance:\n{counts / len(df) * 100}')

In [ ]:
# Numeric feature distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'risk_label' in numeric_cols:
    numeric_cols.remove('risk_label')

n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='#2196F3', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Numeric Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 8))
corr = df[numeric_cols + ['risk_label']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: numeric features vs risk_label
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='risk_label', ax=axes[i])
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('Risk Label (0=Low, 1=High)')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature vs Risk Label', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
from data_preprocessing import clean_data, encode_features, engineer_features, split_data

df_clean = clean_data(df.copy())
df_clean = encode_features(df_clean)
df_clean = engineer_features(df_clean)

print(f'Final shape after preprocessing: {df_clean.shape}')
df_clean.head()

In [ ]:
X_train, X_test, y_train, y_test, scaler = split_data(df_clean)
print(f'Train size : {X_train.shape[0]}')
print(f'Test size  : {X_test.shape[0]}')

## 4. Model Training

In [ ]:
# Logistic Regression with GridSearchCV
lr_params = {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']}
lr_gs = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    lr_params, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
lr_gs.fit(X_train, y_train)
lr_best = lr_gs.best_estimator_
print(f'LR best params : {lr_gs.best_params_}')
print(f'LR CV AUC      : {lr_gs.best_score_:.4f}')

In [ ]:
# Random Forest with GridSearchCV
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}
rf_gs = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_params, cv=5, scoring='roc_auc', n_jobs=-1, verbose=0
)
rf_gs.fit(X_train, y_train)
rf_best = rf_gs.best_estimator_
print(f'RF best params : {rf_gs.best_params_}')
print(f'RF CV AUC      : {rf_gs.best_score_:.4f}')

## 5. Evaluation

In [ ]:
for name, model in [('Logistic Regression', lr_best), ('Random Forest', rf_best)]:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))
    print(f'  ROC-AUC: {auc:.4f}')

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, [('Logistic Regression', lr_best), ('Random Forest', rf_best)]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Low Risk', 'High Risk'],
        yticklabels=['Low Risk', 'High Risk'],
        ax=ax
    )
    ax.set_title(f'Confusion Matrix — {name}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ROC Curve comparison
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#2196F3', '#4CAF50']
for (name, model), color in zip([('Logistic Regression', lr_best), ('Random Forest', rf_best)], colors):
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../reports/figures/roc_curve_comparison.png', dpi=150)
plt.show()

In [ ]:
# Feature importance — Random Forest
feature_names = [c for c in df_clean.columns if c != 'risk_label']
importances = rf_best.feature_importances_
indices = np.argsort(importances)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(15), importances[indices], color='#2196F3', edgecolor='white')
ax.set_xticks(range(15))
ax.set_xticklabels([feature_names[i] for i in indices], rotation=40, ha='right')
ax.set_ylabel('Mean impurity decrease')
ax.set_title('Top 15 Feature Importances — Random Forest')
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150)
plt.show()

## 6. Save Models

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(lr_best, '../models/logistic_regression.pkl')
joblib.dump(rf_best, '../models/random_forest.pkl')
joblib.dump(scaler,  '../models/scaler.pkl')
print('Models saved to models/')